# Notebook — Khởi tạo 3 Controller VM trên Stack1 theo netplan mới của thầy

**Mục tiêu:** tạo lại 3 Controller VM trên **Server A / stack1 / 192.168.150.9**.

Điểm sửa quan trọng theo ảnh thầy gửi:

- NIC `enp1s0` → không đặt IP trực tiếp, làm slave cho `br-mgmt`.
- NIC `enp2s0` → nhận Lab IP `192.168.150.21/22/23` và giữ **default route ra Internet** qua `192.168.150.254`.
- Bridge `br-mgmt` → IP nội bộ `10.0.0.11/12/13`, MTU `1450`.
- `br-mgmt` chỉ dùng route nội bộ `10.0.0.0/8`, **không dùng làm default gateway**.

| VM | Hostname | br-mgmt | Lab IP |
|---|---|---:|---:|
| controller-vm-1 | controller-1 | 10.0.0.11/24 | 192.168.150.21/24 |
| controller-vm-2 | controller-2 | 10.0.0.12/24 | 192.168.150.22/24 |
| controller-vm-3 | controller-3 | 10.0.0.13/24 | 192.168.150.23/24 |

## 0. Kiểm tra chạy trên Stack1
Chỉ chạy notebook này trên **stack1 / Server A** vì đây là host duy nhất còn chạy `libvirtd` để quản lý 3 Controller VM.

In [ ]:
hostname
ip -br addr
systemctl status libvirtd --no-pager
virsh list --all

## 1. Kiểm tra bridge host cần có
Trên stack1 phải có ít nhất:

- `br-lab` — mạng lab/uplink 192.168.150.0/24
- `br-mgmt` — mạng nội bộ OpenStack 10.0.0.0/24, MTU 1450

In [ ]:
ip -br addr show br-lab
ip -br addr show br-mgmt
bridge link | egrep 'br-lab|br-mgmt|vxlan100' || true

## 2. Dọn 3 VM cũ nếu còn sót
Chạy cell này nếu thầy đã xóa VM rồi thì không sao; lệnh có `|| true` để tránh lỗi.

In [ ]:
for vm in controller-vm-1 controller-vm-2 controller-vm-3; do
  virsh destroy $vm 2>/dev/null || true
  virsh undefine $vm --nvram --remove-all-storage 2>/dev/null || true
done

for lv in ctrl-vm-1 ctrl-vm-2 ctrl-vm-3; do
  sudo lvremove -y /dev/vg-lab/$lv 2>/dev/null || true
done

virsh list --all
sudo lvs

## 3. Chuẩn bị thư mục template và cloud-init

In [ ]:
sudo mkdir -p /var/lib/libvirt/templates
sudo chown -R $USER:$USER /var/lib/libvirt/templates
mkdir -p ~/cloud-init-controller

## 4. Tải Ubuntu 24.04 cloud image
Bỏ qua nếu file đã có sẵn.

In [ ]:
cd /var/lib/libvirt/templates

if [ ! -f ubuntu-24.04-base.img ]; then
  wget -c https://cloud-images.ubuntu.com/noble/current/noble-server-cloudimg-amd64.img     -O ubuntu-24.04-base.img
fi

qemu-img info ubuntu-24.04-base.img

## 5. Tạo template controller 100G

In [ ]:
cd /var/lib/libvirt/templates
rm -f tpl-controller.img
cp ubuntu-24.04-base.img tpl-controller.img
qemu-img resize tpl-controller.img 100G
qemu-img info tpl-controller.img | grep 'virtual size'

## 6. Cài package vào template
Nếu mạng lab không ổn, cell này có thể lâu. Mục tiêu là template đã có Docker, cloud-init, chrony và Kolla-Ansible venv.

In [ ]:
cd /var/lib/libvirt/templates

sudo virt-customize -a tpl-controller.img   --update   --install curl,wget,git,vim,htop,net-tools,openssh-server,cloud-init,chrony,python3-pip,python3-venv,python3-dev,build-essential,libssl-dev,libffi-dev,ceph-common   --run-command "curl -fsSL https://get.docker.com | sh"   --run-command "systemctl enable ssh"   --run-command "systemctl enable docker"   --run-command "systemctl enable chrony"   --run-command "usermod -aG docker ubuntu"   --run-command "echo 'ubuntu ALL=(ALL) NOPASSWD:ALL' >> /etc/sudoers"   --run-command "python3 -m venv /opt/kolla-venv"   --run-command "/opt/kolla-venv/bin/pip install -U pip wheel setuptools"   --run-command "/opt/kolla-venv/bin/pip install kolla-ansible"   --timezone Asia/Ho_Chi_Minh

## 7. Cấu hình password và SSH key vào template
Password lab: `123.abc`.

In [ ]:
[ -f ~/.ssh/id_ed25519.pub ] || ssh-keygen -t ed25519 -N "" -f ~/.ssh/id_ed25519

cd /var/lib/libvirt/templates
sudo virt-customize -a tpl-controller.img   --run-command 'sed -i "s/^#PasswordAuthentication.*/PasswordAuthentication yes/" /etc/ssh/sshd_config'   --run-command 'sed -i "s/^KbdInteractiveAuthentication.*/KbdInteractiveAuthentication yes/" /etc/ssh/sshd_config'   --password "ubuntu:password:123.abc"   --root-password "password:123.abc"   --ssh-inject "ubuntu:file:$HOME/.ssh/id_ed25519.pub"

## 8. Sysprep template
Reset machine-id và trạng thái cloud-init trước khi clone.

In [ ]:
cd /var/lib/libvirt/templates
sudo virt-sysprep -a tpl-controller.img --operations defaults,-ssh-hostkeys
sudo chmod 444 tpl-controller.img
ls -lh tpl-controller.img

# PHẦN CLOUD-INIT — Theo netplan mới của thầy

Mẫu netplan bên trong VM sẽ là:

- `enp1s0`: slave của `br-mgmt`, không IP.
- `enp2s0`: Lab IP + default gateway `192.168.150.254` + DNS.
- `br-mgmt`: IP `10.0.0.x`, MTU 1450, route nội bộ `10.0.0.0/8`.

## 9. Tạo cloud-init cho controller-1

In [ ]:
cd ~/cloud-init-controller

cat > ctrl1-user-data << 'EOF'
#cloud-config
hostname: controller-1
fqdn: controller-1.openstack.lab
manage_etc_hosts: true
password: 123.abc
chpasswd: {expire: false}
ssh_pwauth: true
write_files:
  - path: /etc/netplan/50-cloud-init.yaml
    permissions: '0600'
    content: |
      network:
        version: 2
        renderer: networkd
        ethernets:
          enp1s0:
            dhcp4: false
            dhcp6: false
          enp2s0:
            dhcp4: false
            dhcp6: false
            addresses: [192.168.150.21/24]
            routes:
              - to: default
                via: 192.168.150.254
            nameservers:
              addresses: [8.8.8.8, 1.1.1.1]
        bridges:
          br-mgmt:
            interfaces: [enp1s0]
            addresses: [10.0.0.11/24]
            mtu: 1450
            routes:
              - to: 10.0.0.0/8
                scope: link
            parameters:
              stp: false
              forward-delay: 0
runcmd:
  - rm -f /etc/netplan/00-installer-config.yaml
  - netplan generate
  - netplan apply
EOF

cat > ctrl1-meta-data << 'EOF'
instance-id: controller-1
local-hostname: controller-1
EOF

cloud-localds ctrl1-seed.iso ctrl1-user-data ctrl1-meta-data

## 10. Tạo cloud-init cho controller-2

In [ ]:
cd ~/cloud-init-controller

cat > ctrl2-user-data << 'EOF'
#cloud-config
hostname: controller-2
fqdn: controller-2.openstack.lab
manage_etc_hosts: true
password: 123.abc
chpasswd: {expire: false}
ssh_pwauth: true
write_files:
  - path: /etc/netplan/50-cloud-init.yaml
    permissions: '0600'
    content: |
      network:
        version: 2
        renderer: networkd
        ethernets:
          enp1s0:
            dhcp4: false
            dhcp6: false
          enp2s0:
            dhcp4: false
            dhcp6: false
            addresses: [192.168.150.22/24]
            routes:
              - to: default
                via: 192.168.150.254
            nameservers:
              addresses: [8.8.8.8, 1.1.1.1]
        bridges:
          br-mgmt:
            interfaces: [enp1s0]
            addresses: [10.0.0.12/24]
            mtu: 1450
            routes:
              - to: 10.0.0.0/8
                scope: link
            parameters:
              stp: false
              forward-delay: 0
runcmd:
  - rm -f /etc/netplan/00-installer-config.yaml
  - netplan generate
  - netplan apply
EOF

cat > ctrl2-meta-data << 'EOF'
instance-id: controller-2
local-hostname: controller-2
EOF

cloud-localds ctrl2-seed.iso ctrl2-user-data ctrl2-meta-data

## 11. Tạo cloud-init cho controller-3

In [ ]:
cd ~/cloud-init-controller

cat > ctrl3-user-data << 'EOF'
#cloud-config
hostname: controller-3
fqdn: controller-3.openstack.lab
manage_etc_hosts: true
password: 123.abc
chpasswd: {expire: false}
ssh_pwauth: true
write_files:
  - path: /etc/netplan/50-cloud-init.yaml
    permissions: '0600'
    content: |
      network:
        version: 2
        renderer: networkd
        ethernets:
          enp1s0:
            dhcp4: false
            dhcp6: false
          enp2s0:
            dhcp4: false
            dhcp6: false
            addresses: [192.168.150.23/24]
            routes:
              - to: default
                via: 192.168.150.254
            nameservers:
              addresses: [8.8.8.8, 1.1.1.1]
        bridges:
          br-mgmt:
            interfaces: [enp1s0]
            addresses: [10.0.0.13/24]
            mtu: 1450
            routes:
              - to: 10.0.0.0/8
                scope: link
            parameters:
              stp: false
              forward-delay: 0
runcmd:
  - rm -f /etc/netplan/00-installer-config.yaml
  - netplan generate
  - netplan apply
EOF

cat > ctrl3-meta-data << 'EOF'
instance-id: controller-3
local-hostname: controller-3
EOF

cloud-localds ctrl3-seed.iso ctrl3-user-data ctrl3-meta-data

## 12. Tạo LVM disk cho 3 Controller VM

In [ ]:
cd /var/lib/libvirt/templates

sudo lvcreate -V 100G --thin -n ctrl-vm-1 vg-lab/thin-pool
sudo lvcreate -V 100G --thin -n ctrl-vm-2 vg-lab/thin-pool
sudo lvcreate -V 100G --thin -n ctrl-vm-3 vg-lab/thin-pool

sudo qemu-img convert -p -O raw tpl-controller.img /dev/vg-lab/ctrl-vm-1
sudo qemu-img convert -p -O raw tpl-controller.img /dev/vg-lab/ctrl-vm-2
sudo qemu-img convert -p -O raw tpl-controller.img /dev/vg-lab/ctrl-vm-3

sudo lvs

## 13. Khởi tạo controller-vm-1
Thứ tự NIC rất quan trọng:

1. NIC đầu tiên `--network bridge=br-mgmt` → trong VM thường là `enp1s0`.
2. NIC thứ hai `--network bridge=br-lab` → trong VM thường là `enp2s0`.

In [ ]:
virt-install   --name controller-vm-1   --memory 10240 --vcpus 2   --disk /dev/vg-lab/ctrl-vm-1,format=raw,bus=virtio   --disk ~/cloud-init-controller/ctrl1-seed.iso,device=cdrom   --os-variant ubuntu24.04   --network bridge=br-mgmt,model=virtio   --network bridge=br-lab,model=virtio   --graphics none   --noautoconsole   --import

## 14. Khởi tạo controller-vm-2

In [ ]:
virt-install   --name controller-vm-2   --memory 10240 --vcpus 2   --disk /dev/vg-lab/ctrl-vm-2,format=raw,bus=virtio   --disk ~/cloud-init-controller/ctrl2-seed.iso,device=cdrom   --os-variant ubuntu24.04   --network bridge=br-mgmt,model=virtio   --network bridge=br-lab,model=virtio   --graphics none   --noautoconsole   --import

## 15. Khởi tạo controller-vm-3

In [ ]:
virt-install   --name controller-vm-3   --memory 10240 --vcpus 2   --disk /dev/vg-lab/ctrl-vm-3,format=raw,bus=virtio   --disk ~/cloud-init-controller/ctrl3-seed.iso,device=cdrom   --os-variant ubuntu24.04   --network bridge=br-mgmt,model=virtio   --network bridge=br-lab,model=virtio   --graphics none   --noautoconsole   --import

## 16. Kiểm tra VM đã chạy

In [ ]:
sleep 60
virsh list --all
virsh domiflist controller-vm-1
virsh domiflist controller-vm-2
virsh domiflist controller-vm-3

## 17. SSH vào từng Controller VM qua Lab IP

In [ ]:
ssh ubuntu@192.168.150.21 'hostname; ip -br addr; ip route; ping -c 2 1.1.1.1'
ssh ubuntu@192.168.150.22 'hostname; ip -br addr; ip route; ping -c 2 1.1.1.1'
ssh ubuntu@192.168.150.23 'hostname; ip -br addr; ip route; ping -c 2 1.1.1.1'

## 18. Kiểm tra br-mgmt giữa 3 Controller VM

In [ ]:
ssh ubuntu@192.168.150.21 'ping -c 3 10.0.0.12; ping -c 3 10.0.0.13'
ssh ubuntu@192.168.150.22 'ping -c 3 10.0.0.11; ping -c 3 10.0.0.13'
ssh ubuntu@192.168.150.23 'ping -c 3 10.0.0.11; ping -c 3 10.0.0.12'

## 19. Kiểm tra Docker và Kolla-Ansible trong VM

In [ ]:
for ip in 192.168.150.21 192.168.150.22 192.168.150.23; do
  echo "===== $ip ====="
  ssh ubuntu@$ip 'hostname; docker --version; /opt/kolla-venv/bin/kolla-ansible --version || true'
done

## 20. Sửa tay netplan nếu cloud-init không ra đúng tên NIC
Nếu vào VM thấy NIC không phải `enp1s0/enp2s0`, kiểm tra bằng:

```bash
ip -br link
```

Sau đó sửa `/etc/netplan/50-cloud-init.yaml` theo đúng tên NIC thật. Mẫu thầy đang dùng là `enp1s0` cho br-mgmt và `enp2s0` cho Lab IP.

In [ ]:
# Chạy trong từng Controller VM nếu cần:
# sudo nano /etc/netplan/50-cloud-init.yaml
# sudo netplan generate
# sudo netplan apply
# ip -br addr
# ip route
# ping -c 2 1.1.1.1